# Phase 2.5 (split A): Logistic Regression + Random Forest

**Group 01-02 | DS261 Spring 2026**

This notebook is the **memory/workflow split** counterpart to [phase2.5_modeling_GBT_undersample.ipynb](phase2.5_modeling_GBT_undersample.ipynb). It loads **`phase2_df_undersample_*_ml.parquet`** from **[Phase2.4_Pipeline_StandardScaler_ML_udersample.ipynb](templates/Final/Phase2.4_Pipeline_StandardScaler_ML_udersample.ipynb)** (Phase 3.3 undersampled train → StandardScaler on non-`*_scl` → ML export). When Phase 2.4 runs with `WRITE_PIPELINE_MODEL=True`, the fitted `PipelineModel` is saved under **`{BASE_GROUP}/phase3_pipeline_model_standard_undersample`** on DBFS.

**Use the all-in-one** [phase2.5_modeling_final.ipynb](phase2.5_modeling_final.ipynb) if you want one run with a combined summary table and enough cluster memory.

**Pipeline context:** Phase 2.4 Notebook **A** exports **`features`** and **`DEP_DEL15`** only (no `class_weight` column). Imbalance is handled upstream by Phase 3.3 train undersampling; val/test keep the natural mix. **LR** and **RF** share the same `df_train_ml` rows; no second undersample in Phase 2.5. F-β (β=2), RF threshold tuning on val, LR grid search, time-series CV for LR.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import time

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array

# ── Configuration ──────────────────────────────────────────────────────────────
BASE_GROUP = "dbfs:/student-groups/Group_01_02"
TARGET     = "DEP_DEL15"
BETA       = 2.0
VERIFY_ROW_COUNTS = False  # True: extra count() diagnostics on load (memory heavy)

# Notebook A (StandardScaler) ML parquets only
ML_PREFIX_A = "phase2_df_undersample"
TRAIN_ML_PATH = f"{BASE_GROUP}/{ML_PREFIX_A}_train_ml.parquet"
VAL_ML_PATH   = f"{BASE_GROUP}/{ML_PREFIX_A}_val_ml.parquet"
TEST_ML_PATH  = f"{BASE_GROUP}/{ML_PREFIX_A}_test_ml.parquet"

print("Libraries loaded (StandardScaler LR/RF split notebook).")
print(f"F-β metric: β = {BETA}  (recall weighted {int(BETA)}× over precision)")



Libraries loaded (StandardScaler LR/RF split notebook).
F-β metric: β = 2.0  (recall weighted 2× over precision)


## 1. Data Loading & Verification

Load ML-ready parquets from **Phase 2.4 Notebook A** Phase2.4_Pipeline_StandardScaler_ML_udersample.ipynb : `features` with StandardScaler (train-fitted) on non-`*_scl` columns and `DEP_DEL15`. 



In [0]:
df_train_ml = spark.read.parquet(TRAIN_ML_PATH).cache()
df_val_ml   = spark.read.parquet(VAL_ML_PATH).cache()
df_test_ml  = spark.read.parquet(TEST_ML_PATH).cache()

if VERIFY_ROW_COUNTS:
    n_train = df_train_ml.count()
    n_val   = df_val_ml.count()
    n_test  = df_test_ml.count()
    print(f"{'='*60}")
    print(f"  DATA SCALE (Notebook A / LR+RF split)")
    print(f"{'='*60}")
    print(f"  Train:      {n_train:>12,} rows")
    print(f"  Validation: {n_val:>12,} rows")
    print(f"  Test:       {n_test:>12,} rows")
    print(f"  Total:      {n_train + n_val + n_test:>12,} rows")
else:
    n_train = df_train_ml.count()
    print("VERIFY_ROW_COUNTS=False: skipping val/test counts on load.")
    print(f"  Train rows (single count): {n_train:,}")

_dim = df_train_ml.first()["features"].size
print(f"  Feature vector size: {_dim}")



VERIFY_ROW_COUNTS=False: skipping val/test counts on load.
  Train rows (single count): 14,515,569
  Feature vector size: 103


In [0]:
# Class balance per split (skipped when VERIFY_ROW_COUNTS=False to avoid extra full scans)
if VERIFY_ROW_COUNTS:
    for name, df in [("Train", df_train_ml), ("Validation", df_val_ml), ("Test", df_test_ml)]:
        total   = df.count()
        delayed = df.filter(F.col(TARGET) == 1).count()
        ontime  = total - delayed
        print(f"  {name:12s}: {total:>12,} | Delayed: {delayed:>10,} ({100*delayed/total:.1f}%) "
              f"| On-time: {ontime:>10,} ({100*ontime/total:.1f}%)")
else:
    print("VERIFY_ROW_COUNTS=False: skipping per-split class balance counts.")



VERIFY_ROW_COUNTS=False: skipping per-split class balance counts.


## 2. Evaluation Helpers — F-β Score (β=2)

All model comparisons use **F-β with β=2** as the primary metric.

$$F_\beta = \frac{(1+\beta^2) \cdot \text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

With β=2 this becomes:
$$F_2 = \frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

AUROC is retained as a secondary metric to assess rank-ordering ability across all thresholds.

In [0]:
def fbeta(precision, recall, beta=BETA):
    """Compute F-β score."""
    denom = beta**2 * precision + recall
    if denom < 1e-12:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def evaluate_model(model, df, label_col=TARGET, threshold=None):
    """
    Evaluate a fitted PySpark ML model.
    If threshold is not None, override the default 0.5 classification threshold.
    Returns a dict of metrics including F-β (primary), F1, AUROC, and confusion matrix.
    """
    predictions = model.transform(df)

    if threshold is not None:
        predictions = predictions.withColumn(
            "prediction",
            F.when(vector_to_array(F.col("probability"))[1] >= threshold, 1.0).otherwise(0.0),
        )

    auroc = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction", labelCol=label_col, metricName="areaUnderROC"
    ).evaluate(predictions)

    tp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 1)).count()
    fp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 0)).count()
    fn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 1)).count()
    tn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 0)).count()

    prec_del = tp / max(tp + fp, 1)
    rec_del  = tp / max(tp + fn, 1)
    f1_del   = 2 * prec_del * rec_del / max(prec_del + rec_del, 1e-12)
    fb_del   = fbeta(prec_del, rec_del)

    accuracy = (tp + tn) / max(tp + fp + fn + tn, 1)

    return {
        f"F{BETA:.0f} (delayed)": round(fb_del, 4),    # PRIMARY
        "F1 (delayed)": round(f1_del, 4),
        "Precision (delayed)": round(prec_del, 4),
        "Recall (delayed)": round(rec_del, 4),
        "AUROC": round(auroc, 4),
        "Accuracy": round(accuracy, 4),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


def print_metrics(metrics, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    for k, v in metrics.items():
        if k in ("TP", "FP", "FN", "TN"):
            print(f"  {k:25s}: {v:>12,}")
        else:
            marker = "  <-- PRIMARY" if f"F{BETA:.0f}" in k and "delayed" in k else ""
            print(f"  {k:25s}: {v:>12.4f}{marker}")

def time_series_cv(df, estimator_fn, n_folds=5, label_col=TARGET, window_frac=0.20):
    """
    Rolling-window time-series cross-validation on training data.

    Unlike expanding-window CV (where train grows each fold), rolling-window
    uses a FIXED-SIZE training window that slides forward in time. This means:
      - Each fold trains on the same amount of data
      - Earlier data is dropped as the window advances
      - Better reflects real-world deployment where only recent data is relevant
      - Prevents early folds from dominating due to larger training sizes

    window_frac : fraction of total rows used for each fold's training window (default 20%)
    Each fold's validation window is the next (1/n_folds * remaining) chunk after train.

    Fold layout example with n_folds=3, window_frac=0.20:
      Fold 1: Train [0%–20%]   → Val [20%–33%]
      Fold 2: Train [13%–33%]  → Val [33%–47%]
      Fold 3: Train [27%–47%]  → Val [47%–60%]
    """
    # Sort by fl_date_sort if available so fold boundaries are truly temporal.
    if "fl_date_sort" in df.columns:
        df_indexed = df.orderBy("fl_date_sort").withColumn("_row_id", F.monotonically_increasing_id()).cache()
    else:
        print("WARNING: 'fl_date_sort' not in DataFrame — fold boundaries may not be temporal. "
              "Re-run Phase 2.4 to add fl_date_sort to ML parquets.")
        df_indexed = df.withColumn("_row_id", F.monotonically_increasing_id()).cache()
    total = df_indexed.count()
    fold_metrics = []

    # Val window size: split remaining data (after first train window) into n_folds chunks
    val_frac = (1.0 - window_frac) / n_folds

    for fold in range(n_folds):
        val_start  = window_frac + fold * val_frac
        val_end    = val_start + val_frac
        train_start = val_start - window_frac  # fixed-size window ending at val_start

        quantiles = df_indexed.approxQuantile(
            "_row_id",
            [train_start, val_start, val_end],
            0.01
        )
        train_lo, train_hi, val_hi = quantiles

        fold_train = df_indexed.filter(
            (F.col("_row_id") >= train_lo) & (F.col("_row_id") < train_hi)
        ).drop("_row_id")

        fold_val = df_indexed.filter(
            (F.col("_row_id") >= train_hi) & (F.col("_row_id") < val_hi)
        ).drop("_row_id")

        n_ft, n_fv = fold_train.count(), fold_val.count()
        if n_fv == 0 or n_ft == 0:
            continue

        print(f"  Fold {fold+1}: Train {n_ft:,} (~{100*window_frac:.0f}% window) "
              f"| Val {n_fv:,}")
        model   = estimator_fn().fit(fold_train)
        metrics = evaluate_model(model, fold_val, label_col)
        metrics["fold"] = fold + 1
        fold_metrics.append(metrics)
        print(f"    F{BETA:.0f}(delayed)={metrics[f'F{BETA:.0f} (delayed)']:.4f}  "
              f"F1={metrics['F1 (delayed)']:.4f}  AUROC={metrics['AUROC']:.4f}")

    df_indexed.unpersist()
    return fold_metrics

print("Evaluation helpers defined.")

Evaluation helpers defined.


## 3. Class Weights

Balanced inverse-frequency weights: class with fewer samples receives a higher weight.
This is computed from the **training set only** to avoid leakage.

In [0]:
has_weight = "class_weight" in df_train_ml.columns
print(f"class_weight column already present: {has_weight}")

if not has_weight:
    total         = n_train
    delayed_count = df_train_ml.filter(F.col(TARGET) == 1).count()
    ontime_count  = total - delayed_count
    w_delayed     = total / (2.0 * delayed_count)
    w_ontime      = total / (2.0 * ontime_count)
    print(f"  w_delayed = {w_delayed:.4f}  (class ~{100*delayed_count/total:.1f}%)")
    print(f"  w_ontime  = {w_ontime:.4f}  (class ~{100*ontime_count/total:.1f}%)")

    for split, dfx in [("train", df_train_ml), ("val", df_val_ml), ("test", df_test_ml)]:
        dfx = dfx.withColumn("class_weight",
            F.when(F.col(TARGET) == 1, w_delayed).otherwise(w_ontime))
        if split == "train":  df_train_ml = dfx.cache()
        elif split == "val":  df_val_ml   = dfx
        else:                 df_test_ml  = dfx
    print("  class_weight added to all splits.")
else:
    weight_by_class = (
        df_train_ml
        .groupBy(TARGET)
        .agg(F.first("class_weight").alias("class_weight"), F.count("*").alias("count"))
        .orderBy(TARGET)
        .collect()
    )
    for row in weight_by_class:
        label = "w_delayed" if row[TARGET] == 1 else "w_ontime"
        pct   = 100 * row["count"] / n_train
        print(f"  {label:12s} (DEP_DEL15={row[TARGET]}): "
              f"weight={row['class_weight']:.4f}  ({row['count']:,} rows, {pct:.1f}%)")

class_weight column already present: False
  w_delayed = 1.9998  (class ~25.0%)
  w_ontime  = 0.6667  (class ~75.0%)
  class_weight added to all splits.


## 4. Train distribution (Phase 3.3 parquet)

Class balance is defined in **Phase 3.3** on `df_train_phase3_downsampled.parquet`. **LR** and **RF** both read the same `df_train_ml` rows from Phase 2.4 Notebook **A**; this section caches that frame as `df_train_us` for RF (alias only — **no** additional sampling here). Val/test stay the natural mix.



In [0]:
n_delayed_train = df_train_ml.filter(F.col(TARGET) == 1).count()
n_ontime_train  = df_train_ml.filter(F.col(TARGET) == 0).count()

print("Train (Notebook A ML parquet = Phase 3.3 downsampled export):")
print(f"  On-time:  {n_ontime_train:>12,}")
print(f"  Delayed:  {n_delayed_train:>12,}")
print(f"  Ratio:    {n_ontime_train/max(n_delayed_train,1):.2f}:1")

df_train_us = df_train_ml.cache()
print(f"\nRF train alias df_train_us — no Phase 2.5 resample: {df_train_us.count():,} rows")



Train (Notebook A ML parquet = Phase 3.3 downsampled export):
  On-time:    10,886,398
  Delayed:     3,629,171
  Ratio:    3.00:1

RF train alias df_train_us — no Phase 2.5 resample: 14,515,569 rows


## 5. Logistic Regression + Grid Search

LR trains on the **same Phase 3.3 train rows** as RF, with **inverse-frequency `class_weight`** (no extra row sampling in this notebook). L1 regularization
drives uninformative coefficients to zero. Grid search over regularization strength and
L1/L2 mix to find the best configuration.

In [0]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol=TARGET,
    weightCol="class_weight",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=1.0,   # L1
    family="binomial",
)

print("Training LR baseline (class-weighted)...")
t0 = time.time()
lr_model = lr.fit(df_train_ml)
lr_time  = time.time() - t0
print(f"Done in {lr_time:.1f}s")

lr_val = evaluate_model(lr_model, df_val_ml)
print_metrics(lr_val, "LR baseline — VALIDATION")

Training LR baseline (class-weighted)...
Done in 346.6s

  LR baseline — VALIDATION
  F2 (delayed)             :       0.5017  <-- PRIMARY
  F1 (delayed)             :       0.3332
  Precision (delayed)      :       0.2136
  Recall (delayed)         :       0.7569
  AUROC                    :       0.6655
  Accuracy                 :       0.5156
  TP                       :      765,915
  FP                       :    2,819,828
  FN                       :      245,979
  TN                       :    2,497,816


In [0]:
# Grid search: 3 × 3 = 9 combos × 3 folds = 27 fits
# CV evaluator uses AUROC (rank-ordering), then we re-evaluate best model with F-β
evaluator_auroc = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction", labelCol=TARGET, metricName="areaUnderROC"
)

lr_gs = LogisticRegression(
    featuresCol="features", labelCol=TARGET, weightCol="class_weight",
    maxIter=100, family="binomial",
)
param_grid_lr = (
    ParamGridBuilder()
    .addGrid(lr_gs.regParam,        [0.001, 0.01, 0.1])
    .addGrid(lr_gs.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
cv_lr = CrossValidator(
    estimator=lr_gs,
    estimatorParamMaps=param_grid_lr,
    evaluator=evaluator_auroc,
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("LR grid search (9 combos × 3 folds = 27 fits)...")
t0 = time.time()
cv_lr_model = cv_lr.fit(df_train_ml)
gs_time     = time.time() - t0
print(f"Done in {gs_time:.1f}s")

best_lr = cv_lr_model.bestModel
print(f"\nBest LR: regParam={best_lr._java_obj.getRegParam():.4f}, "
      f"elasticNetParam={best_lr._java_obj.getElasticNetParam():.2f}")

print("\nAll grid search results (CV AUROC):")
for params, score in zip(param_grid_lr, cv_lr_model.avgMetrics):
    rp = params[lr_gs.regParam]
    en = params[lr_gs.elasticNetParam]
    print(f"  regParam={rp}, elasticNet={en} -> CV AUROC={score:.4f}")

best_lr_val = evaluate_model(best_lr, df_val_ml)
print_metrics(best_lr_val, "Best LR (grid search) — VALIDATION")

LR grid search (9 combos × 3 folds = 27 fits)...


In [0]:
# LR threshold sweep on validation (same grid as RF; maximises F-β with global BETA)
val_preds_lr = best_lr.transform(df_val_ml).cache()

THRESHOLDS_LR = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("LR Threshold Sweep (validation set):")
print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

lr_thresh_results = []
best_lr_thresh, best_lr_fb = 0.5, 0.0

for thresh in THRESHOLDS_LR:
    preds = val_preds_lr.withColumn(
        "pred_adj",
        F.when(vector_to_array(F.col("probability"))[1] >= thresh, 1.0).otherwise(0.0),
    )
    tp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 1)).count()
    fp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 0)).count()
    fn = preds.filter((F.col("pred_adj") == 0) & (F.col(TARGET) == 1)).count()
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    fb = fbeta(prec, rec)
    lr_thresh_results.append({"threshold": thresh, "precision": prec, "recall": rec, "f1": f1, f"f{BETA:.0f}": fb})
    marker = " <-- best" if fb > best_lr_fb else ""
    if fb > best_lr_fb:
        best_lr_thresh, best_lr_fb = thresh, fb
    print(f"  {thresh:>8.2f}  {prec:>10.4f}  {rec:>10.4f}  {f1:>10.4f}  {fb:>10.4f}{marker}")

print(f"\nBest LR threshold: {best_lr_thresh} (F{BETA:.0f}={best_lr_fb:.4f})")
val_preds_lr.unpersist()

lr_val_tuned = evaluate_model(best_lr, df_val_ml, threshold=best_lr_thresh)
print_metrics(lr_val_tuned, f"LR (threshold={best_lr_thresh}) — VALIDATION")

## 6. Random Forest — Phase 3.3 train + Threshold Tuning

Random Forest is trained on **`df_train_us`** (same rows as `df_train_ml`, with `weightCol`). We then sweep thresholds
on the validation set to find the one that maximises **F-β (β=2)**.

In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=TARGET,
    # weightCol omitted: df_train_us comes from Phase 3.3 downsampled parquet;
    # combining undersampling + weightCol double-corrects for imbalance.
    numTrees=100,
    maxDepth=10,
    minInstancesPerNode=100,
    subsamplingRate=0.8,
    featureSubsetStrategy="sqrt",
    seed=42,
)

print("Training RF baseline (Phase 3.3 train distribution, no weightCol)...")
t0 = time.time()
rf_model = rf.fit(df_train_us)
rf_time  = time.time() - t0
print(f"Done in {rf_time:.1f}s")

rf_val_default = evaluate_model(rf_model, df_val_ml)
print_metrics(rf_val_default, "RF baseline (threshold=0.5) — VALIDATION")

# ── RF Grid Search ─────────────────────────────────────────────────────────────
rf_gs = RandomForestClassifier(
    featuresCol="features",
    labelCol=TARGET,
    minInstancesPerNode=100,
    featureSubsetStrategy="sqrt",
    seed=42,
)
param_grid_rf = (
    ParamGridBuilder()
    .addGrid(rf_gs.numTrees,        [100, 200])
    .addGrid(rf_gs.maxDepth,        [8, 12, 16])
    .addGrid(rf_gs.subsamplingRate, [0.7, 0.9])
    .build()
)
cv_rf = CrossValidator(
    estimator=rf_gs,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator_auroc,
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("\nRF grid search (12 combos × 3 folds = 36 fits) on Phase 3.3 train...")
t0 = time.time()
cv_rf_model = cv_rf.fit(df_train_us)
rf_gs_time  = time.time() - t0
print(f"Done in {rf_gs_time:.1f}s")

rf_model = cv_rf_model.bestModel
print(f"\nBest RF: numTrees={rf_model._java_obj.getNumTrees()}, "
      f"maxDepth={rf_model._java_obj.getMaxDepth()}, "
      f"subsamplingRate={rf_model._java_obj.getSubsamplingRate():.2f}")

print("\nAll RF grid search results (CV AUROC):")
for params, score in zip(param_grid_rf, cv_rf_model.avgMetrics):
    nt = params[rf_gs.numTrees]
    md = params[rf_gs.maxDepth]
    sr = params[rf_gs.subsamplingRate]
    print(f"  numTrees={nt}, maxDepth={md}, subsamplingRate={sr} -> CV AUROC={score:.4f}")

rf_val_default = evaluate_model(rf_model, df_val_ml)
print_metrics(rf_val_default, "Best RF (grid search, threshold=0.5) — VALIDATION")

### RF Threshold Tuning

We sweep probability thresholds from 0.10 to 0.50 on the **validation set** and select
the threshold that maximises **F-β (β=2)**. Threshold tuning is performed **after**
training — it does not touch the test set.

In [0]:
val_preds_rf = rf_model.transform(df_val_ml).cache()

THRESHOLDS = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print(f"RF Threshold Sweep (validation set):")
print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

rf_thresh_results = []
best_rf_thresh, best_rf_fb = 0.5, 0.0

for thresh in THRESHOLDS:
    preds = val_preds_rf.withColumn(
        "pred_adj",
        F.when(vector_to_array(F.col("probability"))[1] >= thresh, 1.0).otherwise(0.0)
    )
    tp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 1)).count()
    fp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 0)).count()
    fn = preds.filter((F.col("pred_adj") == 0) & (F.col(TARGET) == 1)).count()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-12)
    fb   = fbeta(prec, rec)
    rf_thresh_results.append({"threshold": thresh, "precision": prec, "recall": rec,
                               "f1": f1, f"f{BETA:.0f}": fb})
    marker = " <-- best" if fb > best_rf_fb else ""
    if fb > best_rf_fb:
        best_rf_thresh, best_rf_fb = thresh, fb
    print(f"  {thresh:>8.2f}  {prec:>10.4f}  {rec:>10.4f}  {f1:>10.4f}  {fb:>10.4f}{marker}")

print(f"\nBest RF threshold: {best_rf_thresh} (F{BETA:.0f}={best_rf_fb:.4f})")
val_preds_rf.unpersist()

### Why we chose this threshold

The threshold sweep above surfaces the precision–recall tradeoff at each cutoff.
We select the threshold that maximises **F-β (β=2)** on the **validation set only** —
the test set is never examined during tuning.

**Interpretation:**
- A low threshold (e.g. 0.10) aggressively flags flights as delayed, capturing high recall
  but generating many false alarms (low precision). F-β penalises this less than F1 due
  to β=2 upweighting recall, so the optimal point sits at a lower threshold than F1 would
  select.
- A high threshold (0.50) is conservative — it trusts the model's default decision
  boundary but misses many genuine delays (lower recall).
- **The selected threshold** is the point where increasing recall further would require
  a precision sacrifice so severe that even the recall-weighted F-β starts declining.

This threshold is held fixed and applied to the blind test set at the end.

In [0]:
rf_val_tuned = evaluate_model(rf_model, df_val_ml, threshold=best_rf_thresh)
print_metrics(rf_val_tuned, f"RF (threshold={best_rf_thresh}) — VALIDATION")

### RF Probability Calibration (Isotonic Regression)

Random Forest vote-averaging produces poorly calibrated probabilities — they cluster
near 0 and 1, making raw threshold comparisons unreliable. We fit an **isotonic
regression** calibrator on the **validation** set probabilities (leak-safe: calibrator
does not see the test set) and re-sweep thresholds on the calibrated scores.

In [0]:
from sklearn.isotonic import IsotonicRegression as _IsoReg

_rf_val_prob_pd = (
    rf_model.transform(df_val_ml)
    .select(
        vector_to_array(F.col("probability"))[1].alias("prob"),
        F.col(TARGET).alias("label"),
    )
    .toPandas()
)

_iso_rf = _IsoReg(out_of_bounds="clip")
_iso_rf.fit(_rf_val_prob_pd["prob"].values, _rf_val_prob_pd["label"].values)

_cal_probs = _iso_rf.predict(_rf_val_prob_pd["prob"].values)
_labels    = _rf_val_prob_pd["label"].values

print("RF Calibrated Threshold Sweep (validation, isotonic calibration):")
print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

best_rf_cal_thresh, best_rf_cal_fb = 0.5, 0.0
for thresh in THRESHOLDS:
    _pred = (_cal_probs >= thresh).astype(int)
    _tp = ((_pred == 1) & (_labels == 1)).sum()
    _fp = ((_pred == 1) & (_labels == 0)).sum()
    _fn = ((_pred == 0) & (_labels == 1)).sum()
    _prec = _tp / max(_tp + _fp, 1)
    _rec  = _tp / max(_tp + _fn, 1)
    _f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
    _fb   = fbeta(_prec, _rec)
    marker = " <-- best" if _fb > best_rf_cal_fb else ""
    if _fb > best_rf_cal_fb:
        best_rf_cal_thresh, best_rf_cal_fb = thresh, _fb
    print(f"  {thresh:>8.2f}  {_prec:>10.4f}  {_rec:>10.4f}  {_f1:>10.4f}  {_fb:>10.4f}{marker}")

print(f"\nBest RF calibrated threshold: {best_rf_cal_thresh} (F{BETA:.0f}={best_rf_cal_fb:.4f})")
print(f"(Compare uncalibrated: threshold={best_rf_thresh}, F{BETA:.0f}={best_rf_fb:.4f})")

### 6b. LR |coefficients| and RF importances (optional)

Use with **`phase3_final_num_cols.json`** (same order as Phase 2.4 **A** assembler) for **hypotheses** and **validation-only** ablation notes — not for tuning on the blind test.

In [0]:
import json

try:
    _raw = dbutils.fs.head(f"{BASE_GROUP}/phase3_final_num_cols.json", max_bytes=1_000_000)
    _names = json.loads(_raw)
except Exception as e:
    print(f"Skipping LR/RF diagnostics: {e}")
    _names = []

if _names:
    _c = best_lr.coefficients.toArray()
    lr_tbl = pd.DataFrame(
        [
            (_names[i] if i < len(_names) else f"feat_{i}", float(_c[i]), abs(float(_c[i])))
            for i in range(len(_c))
        ],
        columns=["feature", "coef", "abs_coef"],
    ).sort_values("abs_coef", ascending=False)

    print("Top 25 LR |coefficient| (best grid LR):")
    display(spark.createDataFrame(lr_tbl.head(25)))

    _ri = rf_model.featureImportances.toArray()
    rf_tbl = pd.DataFrame(
        [
            (_names[i] if i < len(_names) else f"feat_{i}", float(_ri[i]))
            for i in range(len(_ri))
        ],
        columns=["feature", "importance"],
    )

    print("Top 25 RF importances:")
    display(spark.createDataFrame(rf_tbl.sort_values("importance", ascending=False).head(25)))
    print("Bottom 25 RF importances:")
    display(spark.createDataFrame(rf_tbl.sort_values("importance").head(25)))

    _csv_rf = f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/feature_importance_rf.csv"
    rf_tbl.sort_values("importance", ascending=False).to_csv(_csv_rf, index=False)
    print(f"Full RF table: {BASE_GROUP}/feature_importance_rf.csv")

## 6c. LR + RF Ensemble

Average the predicted delay probabilities from the best LR and best RF models.
Ensemble averaging reduces variance and typically outperforms either model alone.
The ensemble threshold is tuned on the **validation** set only.

In [0]:
_lr_probs_pd = (
    best_lr.transform(df_val_ml)
    .select(vector_to_array(F.col("probability"))[1].alias("p_lr"), F.col(TARGET).alias("label"))
    .toPandas()
)
_rf_probs_pd = (
    rf_model.transform(df_val_ml)
    .select(vector_to_array(F.col("probability"))[1].alias("p_rf"))
    .toPandas()
)

_ens_probs = 0.5 * _lr_probs_pd["p_lr"].values + 0.5 * _rf_probs_pd["p_rf"].values
_ens_labels = _lr_probs_pd["label"].values

print("LR+RF Ensemble Threshold Sweep (validation, equal weights):")
print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

best_ens_thresh, best_ens_fb = 0.5, 0.0
for thresh in THRESHOLDS:
    _pred = (_ens_probs >= thresh).astype(int)
    _tp = ((_pred == 1) & (_ens_labels == 1)).sum()
    _fp = ((_pred == 1) & (_ens_labels == 0)).sum()
    _fn = ((_pred == 0) & (_ens_labels == 1)).sum()
    _prec = _tp / max(_tp + _fp, 1)
    _rec  = _tp / max(_tp + _fn, 1)
    _f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
    _fb   = fbeta(_prec, _rec)
    marker = " <-- best" if _fb > best_ens_fb else ""
    if _fb > best_ens_fb:
        best_ens_thresh, best_ens_fb = thresh, _fb
    print(f"  {thresh:>8.2f}  {_prec:>10.4f}  {_rec:>10.4f}  {_f1:>10.4f}  {_fb:>10.4f}{marker}")

print(f"\nBest ensemble threshold: {best_ens_thresh} (F{BETA:.0f}={best_ens_fb:.4f})")
print(f"(LR alone: F{BETA:.0f}={best_lr_fb:.4f} | RF alone: F{BETA:.0f}={best_rf_fb:.4f})")

# ── Ensemble blind test ────────────────────────────────────────────────────────
_lr_test_pd = (
    best_lr.transform(df_test_ml)
    .select(vector_to_array(F.col("probability"))[1].alias("p_lr"), F.col(TARGET).alias("label"))
    .toPandas()
)
_rf_test_pd = (
    rf_model.transform(df_test_ml)
    .select(vector_to_array(F.col("probability"))[1].alias("p_rf"))
    .toPandas()
)
_ens_test_probs  = 0.5 * _lr_test_pd["p_lr"].values + 0.5 * _rf_test_pd["p_rf"].values
_ens_test_labels = _lr_test_pd["label"].values
_pred_test = (_ens_test_probs >= best_ens_thresh).astype(int)
_tp = ((_pred_test == 1) & (_ens_test_labels == 1)).sum()
_fp = ((_pred_test == 1) & (_ens_test_labels == 0)).sum()
_fn = ((_pred_test == 0) & (_ens_test_labels == 1)).sum()
_tn = ((_pred_test == 0) & (_ens_test_labels == 0)).sum()
_prec = _tp / max(_tp + _fp, 1)
_rec  = _tp / max(_tp + _fn, 1)
_f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
_fb   = fbeta(_prec, _rec)
print(f"\nLR+RF Ensemble (thresh={best_ens_thresh}) — BLIND TEST")
print(f"  F{BETA:.0f} (delayed):        {_fb:.4f}  <-- PRIMARY")
print(f"  F1 (delayed):          {_f1:.4f}")
print(f"  Precision (delayed):   {_prec:.4f}")
print(f"  Recall (delayed):      {_rec:.4f}")
print(f"  TP={_tp:,}  FP={_fp:,}  FN={_fn:,}  TN={_tn:,}")

## 7. Time-Series Cross-Validation

Rolling-window CV validates that model performance is stable across time periods.

In [0]:
best_lr_rp = best_lr._java_obj.getRegParam()
best_lr_en = best_lr._java_obj.getElasticNetParam()

print(f"=== LR Time-Series CV (5 folds, rolling window) ===")
print(f"    Using: regParam={best_lr_rp:.4f}, elasticNetParam={best_lr_en:.2f}")
lr_cv_results = time_series_cv(
    df_train_ml,
    lambda: LogisticRegression(
        featuresCol="features", labelCol=TARGET, weightCol="class_weight",
        maxIter=100, regParam=best_lr_rp, elasticNetParam=best_lr_en, family="binomial",
    )
)
lr_cv_df = pd.DataFrame(lr_cv_results)
fb_col   = f"F{BETA:.0f} (delayed)"
print(f"\nLR CV: F{BETA:.0f}(delayed) = {lr_cv_df[fb_col].mean():.4f} ± {lr_cv_df[fb_col].std():.4f}")
print(f"       AUROC           = {lr_cv_df['AUROC'].mean():.4f} ± {lr_cv_df['AUROC'].std():.4f}")

## 8. Blind Test Evaluation

Final evaluation on the held-out test set. Thresholds selected on validation are applied
here. The test set is examined **once** — these are the reported Phase 3 results.

In [0]:
print("Evaluating on blind test set (LR + RF only)...\n")

test_best_lr = evaluate_model(best_lr, df_test_ml, threshold=best_lr_thresh)
test_rf      = evaluate_model(rf_model, df_test_ml, threshold=best_rf_thresh)

print_metrics(test_best_lr, f"Best LR (grid search, thresh={best_lr_thresh}) — BLIND TEST")
print_metrics(test_rf,      f"RF (thresh={best_rf_thresh})  — BLIND TEST")


## 9. Summary Table

In [0]:
all_results = []

model_configs = [
    ("LR (grid search best)", best_lr, df_val_ml, df_test_ml, None, gs_time),
    (f"RF (thresh={best_rf_thresh})", rf_model, df_val_ml, df_test_ml, best_rf_thresh, rf_time),
]

for model_name, model, val_df, test_df, thresh, t_time in model_configs:
    v = evaluate_model(model, val_df, threshold=thresh)
    t = evaluate_model(model, test_df, threshold=thresh)
    for split_name, metrics in [("Validation", v), ("Test (Blind)", t)]:
        row = {"Model": model_name, "Split": split_name, **metrics,
               "Train Time (s)": round(t_time, 1) if split_name == "Validation" else ""}
        all_results.append(row)

summary_df = pd.DataFrame(all_results)
print("\n" + "=" * 90)
print("  LR+RF SPLIT — RESULTS SUMMARY (primary metric: F-β, β=2)")
print("=" * 90)
display(spark.createDataFrame(summary_df))



In [0]:
summary_df.to_csv(f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/phase3_results_summary_lr_rf.csv", index=False)
print(f"Results saved to {BASE_GROUP}/phase3_results_summary_lr_rf.csv")

In [0]:
# Save trained models to DBFS for cross-notebook ensemble
WRITE_MODELS = True  # Set False to skip

if WRITE_MODELS:
    _model_base = f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/phase3_models"
    best_lr.write().overwrite().save(f"{_model_base}/best_lr_ss")
    rf_model.write().overwrite().save(f"{_model_base}/rf_model_ss")
    print(f"Saved: best_lr_ss, rf_model_ss → {_model_base}/")
    print("Load in phase2.5_ensemble.py for cross-notebook ensemble.")



## 10. Key Findings (LR + RF split)

### Train rows and imbalance handling

**LR** and **RF** both train on the **Phase 3.3** downsampled parquet (Notebook **A** ML export). **LR** uses **`class_weight`**; **RF** uses **`weightCol`** on the **same rows** — there is **no** second undersample in Phase 2.5. **RF** threshold tuning on validation maximises **F-β**.

### F-β (β=2)

Threshold sweeps for RF optimise **F-β** on validation; LR uses the default 0.5 decision boundary unless you extend this notebook.

### GBT and combined tables

**GBT**, grid-search details, and **feature importances** (top/bottom + CSV) live in [phase2.5_modeling_GBT_undersample.ipynb](phase2.5_modeling_GBT_undersample.ipynb). Optional **LR |coef|** and **RF importances** in §6b use `phase3_final_num_cols.json` — for hypotheses and **validation** ablation notes, not blind-test tuning. For one consolidated metrics table, run [phase2.5_modeling_final.ipynb](phase2.5_modeling_final.ipynb) or merge `phase3_results_summary_lr_rf.csv` and `phase3_results_summary_gbt.csv`.

